#### Perguntas

- Qual a tendência dos dados?
- Quais os países com maior taxa de mortalidade e de recuperação?
- Como os casos estão distribuídos pelo mundo?

## Bibliotecas

In [1]:
!pip install dash

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 26.1 MB/s eta 0:00:00


In [24]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

import warnings
warnings.filterwarnings('ignore')

## Pré-Processamento

In [25]:
#Import data

df = pd.read_csv(r"countries-aggregated.csv")
df2 = pd.read_csv(r"reference.csv")
cont = pd.read_json("country-by-continent.json")

#Pre processing dataframe

df.rename(columns={"Confirmed": "Confirmed2","Recovered":"Recovered2","Deaths":"Deaths2"}, inplace=True)

df['Date'] = pd.PeriodIndex(df['Date'], freq="M")

columns = ["Confirmed2", "Recovered2","Deaths2"]

df = df.groupby(['Country', 'Date']).last().reset_index()

for i in columns:
    lista = []
    for c in df['Country'].unique():
        lst = [0]
        lst.extend(df[df['Country'] == c][i].tolist()[:-1])
        lista.extend(lst)
    df[i[:-1]] = (df[i] - lista).clip(lower=0)

df = df.drop(columns, axis=1)

#Adding columns to the dataframe

df = df.assign(iso3 = np.zeros(len(df)))
df = df.assign(population = np.zeros(len(df)))

for c in df['Country'].unique():
    idx = np.where(df['Country'] == c)[0]

    iso = df2[df2['Combined_Key'] == c]['iso3'].unique()[0]
    df['iso3'][idx] =  iso

    df['population'][idx] =  sum(df2[df2['iso3'] == iso]['Population'].fillna(0))


#Removing countrys with 0 population

nulls = np.where(df['population']== 0)[0]
df = df.drop(index=nulls)

continentes = []

for c in df['Country'].unique():
    country = cont[cont['country'] == c]['continent'].values[0]
    continentes.extend([country for i in range(28)])

df['Continent'] = continentes


#Data normalization

df['Confirmed_normalized'] = df['Confirmed'] / df['population']
df['Recovered_normalized'] = df['Recovered'] / df['population']
df['Deaths_normalized'] = df['Deaths'] / df['population']


#Chaning the data format

df['Date'] = df['Date'].dt.strftime('%Y-%m')

In [26]:
df

,Country,Date,Confirmed,Recovered,Deaths,iso3,population,Continent,Confirmed_normalized,Recovered_normalized,Deaths_normalized
0,Afghanistan,2020-01,0,0,0,AFG,38928341.0,Asia,0.000000e+00,0.000000e+00,0.000000e+00
1,Afghanistan,2020-02,5,0,0,AFG,38928341.0,Asia,1.284411e-07,0.000000e+00,0.000000e+00
2,Afghanistan,2020-03,161,5,4,AFG,38928341.0,Asia,4.135804e-06,1.284411e-07,1.027529e-07
3,Afghanistan,2020-04,1661,255,56,AFG,38928341.0,Asia,4.266814e-05,6.550497e-06,1.438541e-06
4,Afghanistan,2020-05,13353,1068,194,AFG,38928341.0,Asia,3.430149e-04,2.743502e-05,4.983516e-06
...,...,...,...,...,...,...,...,...,...,...,...
5539,Zimbabwe,2021-12,78633,0,297,ZWE,14862927.0,Africa,5.290546e-03,0.000000e+00,1.998261e-05
5540,Zimbabwe,2022-01,16408,0,334,ZWE,14862927.0,Africa,1.103955e-03,0.000000e+00,2.247202e-05
5541,Zimbabwe,2022-02,6714,0,57,ZWE,14862927.0,Africa,4.517280e-04,0.000000e+00,3.835045e-06
5542,Zimbabwe,2022-03,9906,0,49,ZWE,14862927.0,Africa,6.664905e-04,0.000000e+00,3.296793e-06


In [27]:
#Second Dataframe

df2 = df.groupby(['Continent', 'Date']).sum()
df2 = df2.reset_index()

#Data normalization on the second DataFrame

df2['Confirmed_normalized'] = df2['Confirmed'] / df2['population']
df2['Recovered_normalized'] = df2['Recovered'] / df2['population']
df2['Deaths_normalized'] = df2['Deaths'] / df2['population']

In [28]:
df2

,Continent,Date,Confirmed,Recovered,Deaths,population,Confirmed_normalized,Recovered_normalized,Deaths_normalized
0,Africa,2020-01,0,0,0,1.505949e+09,0.000000e+00,0.000000e+00,0.000000e+00
1,Africa,2020-02,3,1,0,1.505949e+09,1.992100e-09,6.640332e-10,0.000000e+00
2,Africa,2020-03,5579,402,197,1.505949e+09,3.704641e-06,2.669413e-07,1.308145e-07
3,Africa,2020-04,31896,11561,1423,1.505949e+09,2.118000e-05,7.676888e-06,9.449192e-07
4,Africa,2020-05,106240,48368,2569,1.505949e+09,7.054688e-05,3.211796e-05,1.705901e-06
...,...,...,...,...,...,...,...,...,...
163,South America,2021-12,825021,0,9977,7.390656e+08,1.116303e-03,0.000000e+00,1.349948e-05
164,South America,2022-01,8806978,0,23574,7.390656e+08,1.191637e-02,0.000000e+00,3.189703e-05
165,South America,2022-02,5621989,0,42248,7.390656e+08,7.606887e-03,0.000000e+00,5.716407e-05
166,South America,2022-03,1851406,0,30021,7.390656e+08,2.505063e-03,0.000000e+00,4.062021e-05


In [29]:
#Data from Portugal

df_portugal = df[df['Country'] == 'Portugal']
df_countries = df[df['Country'] != 'Portugal'].reset_index(drop=True)

lst = []
for d in df_countries['Date'].unique():
    list_values = df_countries[df_countries['Date'] == d]['Confirmed_normalized'].tolist()
    portugal_value = df_portugal[df_portugal['Date'] == d]['Confirmed_normalized'].iloc[0]
    lst.append([abs(portugal_value - i) for i in list_values])

distances = sorted(list(enumerate([sum(coluna) for coluna in zip(*lst)])), key=lambda x: x[1])

print('3 Países com a evolução mais parecida à de Portugal:', [df_countries['Country'].unique()[i] for i, v in distances[:3]])
print('3 Países com a evolução menos parecida à de Portugal:', [df_countries['Country'].unique()[i] for i, v in distances[-3:]])

3 Países com a evolução mais parecida à de Portugal: ['Monaco', 'France', 'Switzerland']
3 Países com a evolução menos parecida à de Portugal: ['Mongolia', 'Brunei', 'Korea, South']


In [30]:
paises_parecidos = ['Portugal', 'Monaco', 'France', 'Switzerland']

#Third Dataframe

df3 = df.drop(columns = ['Confirmed','Recovered','Deaths', 'iso3', 'population', 'Continent','Recovered_normalized','Deaths_normalized'])
df3['rank'] = df3.groupby('Date')['Confirmed_normalized'].rank(ascending = False, method='min')
df3 = df3[df3['Country'].isin(paises_parecidos)].reset_index(drop = True)

paises_naoparecidos = ['Portugal', 'Mongolia', 'Brunei', 'Korea, South']

#Forth Dataframe

df4 = df.drop(columns = ['Confirmed','Recovered','Deaths', 'iso3', 'population', 'Continent','Recovered_normalized','Deaths_normalized'])
df4['rank'] = df4.groupby('Date')['Confirmed_normalized'].rank(ascending = False, method='min')
df4 = df4[df4['Country'].isin(paises_naoparecidos)].reset_index(drop = True)

In [31]:
df3

,Country,Date,Confirmed_normalized,rank
0,France,2020-01,7.662854e-08,10.0
1,France,2020-02,1.455942e-06,16.0
2,France,2020-03,7.997107e-04,13.0
3,France,2020-04,1.786671e-03,9.0
4,France,2020-05,3.389127e-04,47.0
...,...,...,...,...
107,Switzerland,2021-12,3.673796e-02,6.0
108,Switzerland,2022-01,1.025954e-01,10.0
109,Switzerland,2022-02,6.717939e-02,15.0
110,Switzerland,2022-03,8.036727e-02,8.0


In [32]:
df4

,Country,Date,Confirmed_normalized,rank
0,Brunei,2020-01,0.000000,24.0
1,Brunei,2020-02,0.000000,58.0
2,Brunei,2020-03,0.000295,28.0
3,Brunei,2020-04,0.000021,132.0
4,Brunei,2020-05,0.000007,155.0
...,...,...,...,...
107,Portugal,2021-12,0.023772,15.0
108,Portugal,2022-01,0.122604,7.0
109,Portugal,2022-02,0.061080,17.0
110,Portugal,2022-03,0.031987,27.0


## Visualização

In [33]:
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='dropdown',
        options=[
            {'label': 'Confirmed Cases', 'value': 'Confirmed_normalized'},
            {'label': 'Number of Recovered', 'value': 'Recovered_normalized'},
            {'label': 'Number of Deaths', 'value': 'Deaths_normalized'}],
              value='Confirmed_normalized'),
              dcc.Graph(id='choropleth')])

@app.callback(Output('choropleth', 'figure'),
              [Input('dropdown', 'value')])

def update_choropleth(selected_value):
    fig = px.choropleth(df,
                        locations="iso3",
                        color=selected_value,
                        hover_name="Country",
                        animation_frame=df['Date'],
                        title = 'Confirmed Cases by month in the Entire World',
                        height=600,
                        color_continuous_scale=px.colors.sequential.Plasma,
                        projection="natural earth")

    fig.update_layout(
        coloraxis_colorbar=dict(
        title="Number"))

    return fig

if __name__ == '__main__':
    app.run_server(debug=True)



<IPython.core.display.Javascript object>

In [34]:
fig = px.bar(df2, x='Date', y='Confirmed_normalized', color='Continent',
             labels={'Confirmed_normalized':'Numero de casos confirmados', 'Date':'Date', 'Continent':'Continent'},
             title='Number of confirmed cases by Continent',
             hover_data=['Deaths'], height=400)

fig.update_xaxes(tickangle=-45)
fig.update_yaxes(showgrid=True, gridcolor='black')

fig.update_layout(plot_bgcolor='white',
                  paper_bgcolor='white',
                  width=1600,
                  height=600)

fig.show()

In [35]:
df_filtered = df[df['Country'].isin(paises_parecidos)]

df_pivot = df_filtered.pivot(index='Date', columns='Country', values='Confirmed_normalized')

df_pivot = df_pivot[paises_parecidos] # Só para manter a posição no gráfico (se não é ordem alfabética)

fig = go.Figure(data=go.Heatmap(
                   z=df_pivot.values,
                   x=df_pivot.columns,
                   y=df_pivot.index,
                   hoverongaps = False,
                   colorbar=dict(title='Confirmed Cases Per Capita')))

fig.update_layout(title={'text': "Confirmed Cases (Portugal and similar countries)"},
                  xaxis_title = "Country",
                  yaxis_title = "Date")

fig.show()

In [39]:
fig = px.line(df3, x='Date', y='rank', color='Country', hover_name='Date')

fig.update_layout(title={'text': "Global Ranking by Confirmed Cases (Portugal and similar countries)"},
                  xaxis_title = "Date",
                  yaxis_title = "Global Ranking")

fig.update_traces(line = dict(dash='solid', width=4),
                  mode='lines',
                  hovertemplate = '<b>Ranking</b><br>Top %{y}')

fig.update_layout(plot_bgcolor = 'white',
                  hovermode = 'x',
                  hoverlabel=dict(bgcolor="white",
                                  font_size=16,
                                  font_family="Rockwell"))

fig.update_xaxes(showline=True, linewidth=1, linecolor='black', gridcolor='black', tickangle=-45)
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', gridcolor='black', autorange="reversed")

fig.show()

fig = px.line(df4, x='Date', y='rank', color='Country', hover_name='Date')

fig.update_layout(title={'text': "Global Ranking by Confirmed Cases (Portugal and least similar countries)"},
                  xaxis_title = "Date",
                  yaxis_title = "Global Ranking")

fig.update_traces(line = dict(dash='solid', width=4),
                  mode='lines',
                  hovertemplate = '<b>Ranking</b><br>Top %{y}')

fig.update_layout(plot_bgcolor = 'white',
                  hovermode = 'x',
                  hoverlabel=dict(bgcolor="white",
                                  font_size=16,
                                  font_family="Rockwell"))

fig.update_xaxes(showline=True, linewidth=1, linecolor='black', gridcolor='black', tickangle=-45)
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', gridcolor='black', autorange="reversed")

fig.show()

In [38]:
df_portugal = df[df['Country'] == 'Portugal']
df_portugal['Date'] = pd.to_datetime(df_portugal['Date'], errors='coerce')

frames = [go.Frame(data=[go.Indicator(mode="gauge+number",
                   value=row['Confirmed'],
                   title={'text': f"Date: {row['Date'].strftime('%Y-%m')}"},
                   gauge={'axis': {'range': [None, df_portugal['Confirmed'].max()]}})]) for _, row in df_portugal.iterrows()]

fig = go.Figure(data=[go.Indicator(mode="gauge+number",
                                   value=df_portugal.iloc[0]['Deaths'],
                                   title={'text': f"Date: {df_portugal.iloc[0]['Date'].strftime('%Y-%m')}"},
                                   gauge={'axis': {'range': [None, df_portugal['Confirmed'].max()]},'bar': {'color': 'blue'}})],
                      layout=go.Layout(updatemenus=[dict(type="buttons",
                                       buttons=[dict(label="Play",
                                       method="animate",
                                       args=[None, {"frame": {"duration": 500, "redraw": True},"fromcurrent": True}])],
                                       showactive=False)],
                      title="Evolution of Confirmed Cases in Portugal"),
                      frames=frames)

fig.show()